# 可选实验：Logistic Regression 的 Logistic Loss

在本次不计分实验中，你将：

- 探索 squared error loss 为什么不适合用于 logistic regression 的优化。
- 探索 logistic loss function。


In [1]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from plt_logistic_loss import  plt_logistic_cost, plt_two_logistic_loss_curves, plt_simple_example
from plt_logistic_loss import soup_bowl, plt_logistic_squared_error
plt.style.use('./deeplearning.mplstyle')

## Logistic Regression 能否使用 Squared Error？

<img align="left" src="./images/C1_W3_SqErrorVsLogistic.png" alt="Squared error 与 logistic loss 的比较" style="width:400px;padding:10px;">

回顾 **linear regression**，我们使用了 **squared error cost function**。对于单变量模型：

$$
J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2 \tag{1}
$$

其中：

$$
f_{w,b}(x^{(i)})=wx^{(i)}+b \tag{2}
$$


Linear regression 的 squared error cost 具有良好的凸性，使用合适的 learning rate 沿负 gradient 方向更新，可以到达最小值。


In [2]:
soup_bowl()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

这种 cost function 在 linear regression 中表现良好，因此自然会想到将它用于 logistic regression。然而，如上图所示，现在 $f_{w,b}(x)$ 包含非线性的 sigmoid：$f_{w,b}(x^{(i)})=\operatorname{sigmoid}(wx^{(i)}+b)$。

下面在之前的示例中加入 sigmoid，并尝试 squared error cost。训练数据如下：


In [3]:
x_train = np.array([0., 1, 2, 3, 4, 5],dtype=np.longdouble)
y_train = np.array([0,  0, 0, 1, 1, 1],dtype=np.longdouble)
plt_simple_example(x_train, y_train)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

下面绘制采用 *squared error cost* 时的曲面：

$$
J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2
$$

其中：

$$
f_{w,b}(x^{(i)})=\operatorname{sigmoid}(wx^{(i)}+b)
$$


In [4]:
plt.close('all')
plt_logistic_squared_error(x_train,y_train)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

这张图虽然很有趣，但其曲面不再像 linear regression 的“碗”那样适合优化。

Logistic regression 需要更适合其非线性输出的 cost function。下面从单个样本的 loss function 开始介绍。


## Logistic Loss Function

<img src="./images/C1_W3_LogisticLoss_a.png" alt="Logistic loss 的定义" style="width:250px;padding:2px;">
<img src="./images/C1_W3_LogisticLoss_b.png" alt="Target 为 1 时的 logistic loss" style="width:250px;padding:2px;">
<img src="./images/C1_W3_LogisticLoss_c.png" alt="Target 为 0 时的 logistic loss" style="width:250px;padding:2px;">


Logistic regression 的 loss function 专门用于 target 为 0 或 1 的 classification 问题。

> **术语说明：** 本课程用 **loss** 衡量单个样本的预测与 target 之间的差异，用 **cost** 汇总整个训练集上的 loss。

单个样本的 logistic loss 定义为：

$$
\operatorname{loss}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}),y^{(i)})=
\begin{cases}
-\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right), & y^{(i)}=1, \\
-\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right), & y^{(i)}=0.
\end{cases}
$$

- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是预测值，$y^{(i)}$ 是 target。
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})=g(\mathbf{w}\cdot\mathbf{x}^{(i)}+b)$，其中 $g$ 为 sigmoid function。

这个 loss 使用两条不同的曲线，分别对应 $y=0$ 和 $y=1$。它们具有所需的性质：预测值趋近于 target 时，loss 趋近于 0；预测值偏离 target 时，loss 会迅速增大。请观察下图。


In [5]:
plt_two_logistic_loss_curves()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

将两种情况放在一起观察，其变化趋势类似 squared error loss 的二次曲线。注意，横轴是 sigmoid 输出 $f_{\mathbf{w},b}$；对于有限输入，它严格位于 0 与 1 之间。


上面的分段定义可以改写为更方便实现的形式：

$$
\operatorname{loss}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}),y^{(i)})
=-y^{(i)}\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)
-(1-y^{(i)})\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)
$$

虽然公式看起来较长，但 $y^{(i)}$ 只能为 0 或 1，代入后就很清楚了。当 $y^{(i)}=0$ 时，第一项为 0：

$$
\begin{aligned}
\operatorname{loss}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}),0)
&=-0\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)-(1-0)\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right) \\
&=-\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right).
\end{aligned}
$$

当 $y^{(i)}=1$ 时，第二项为 0：

$$
\begin{aligned}
\operatorname{loss}(f_{\mathbf{w},b}(\mathbf{x}^{(i)}),1)
&=-1\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right)-(1-1)\log\left(1-f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right) \\
&=-\log\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\right).
\end{aligned}
$$

将全部样本的 logistic loss 汇总，就能构造 cost function，这将是下一个实验的主题。现在先观察这个简单例子中，cost 随参数变化的曲面。


In [ ]:
plt.close('all')
cst = plt_logistic_cost(x_train,y_train)

这个曲面更适合 gradient descent：它不再出现前面 squared error 示例中的复杂起伏，并且没有非全局的局部极小值或不连续点。它的形状并非 squared error 对应的标准碗状。

图中同时绘制了 cost 和 cost 的对数，以便看清 cost 很小时仍然存在下降趋势。你可以用鼠标旋转上方的图表。


## 恭喜完成实验！

你已经：

- 理解了 squared error loss 用于 logistic regression 优化时的局限。
- 构建并探索了更适合该 classification 模型的 logistic loss function。
